In [ ]:
#!pip install pymrio numpy pandas matplotlib seaborn ipywidgets plotly

In [ ]:
# Using pymrio library
import pymrio

# Download latest EXIOBASE (includes physical extensions)
exio = pymrio.download_exiobase3(
    storage_folder='D:/Programming/databases',
    system='pxp',  # Product-by-product is usually best for physical flows
    years=[2022]
)

# The physical extensions are stored in:
# exio.satellite.F  # Factor table (physical units per monetary unit)
# exio.satellite.S  # Satellite accounts (total physical flows)

In [2]:
import pymrio
import numpy as np
import pandas as pd

# Load EXIOBASE
exio = pymrio.parse_exiobase3('D:/Programming/databases/IOT_2022_pxp')

In [4]:
# View available extensions
print("Available extensions:")
print(exio.get_extensions())

# Access a specific extension (e.g., emissions)
emissions = exio.emissions

Available extensions:
<generator object IOSystem.get_extensions at 0x000001DBB841AA40>


AttributeError: 'IOSystem' object has no attribute 'emissions'

In [6]:
# Check what's actually available in the parsed object
print("Available attributes:")
print(dir(exio))

Available attributes:
['A', 'B', 'DataFrames', 'G', 'L', 'Y', 'Y_categories', 'Z', '__basic__', '__class__', '__coefficients__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__firstlineno__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__non_agg_attributes__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__static_attributes__', '__str__', '__subclasshook__', '__weakref__', '_apply_extension_method', 'aggregate', 'aggregate_duplicates', 'air_emissions', 'calc_all', 'calc_extensions', 'calc_system', 'contains', 'copy', 'employment', 'empty', 'extension_characterize', 'extension_concate', 'extension_contains', 'extension_convert', 'extension_extract', 'extension_fullmatch', 'extension_match', 'extensions', 'extensions_instance_names', 'factor_inputs', 'find', 'fullmatch', 'get_DataFrame', 'get_Y_categories', 'get_extensio

In [7]:
# The extensions are stored differently - check the structure
print("\nExtensions attribute:")
print(hasattr(exio, 'extensions'))


Extensions attribute:
True


In [8]:
# View the core IO tables that ARE loaded
print("\nCore tables available:")
print(f"Z (transactions): {exio.Z.shape if hasattr(exio, 'Z') else 'Not loaded'}")
print(f"Y (final demand): {exio.Y.shape if hasattr(exio, 'Y') else 'Not loaded'}")
print(f"x (total output): {exio.x.shape if hasattr(exio, 'x') else 'Not loaded'}")


Core tables available:
Z (transactions): (9800, 9800)
Y (final demand): (9800, 343)
x (total output): (9800, 1)


In [9]:
# Alternative approach - load satellite accounts separately
import os

# Check what files are actually in your directory
data_path = 'D:/Programming/databases/IOT_2022_pxp'
print("Files in directory:")
for file in os.listdir(data_path):
    print(file)

Files in directory:
A.txt
air_emissions
employment
factor_inputs
file_parameters.json
IOT_2022_pxp.zip
material
metadata.json
nutrients
unit.txt
water
x.txt
Y.txt
Z.txt


In [12]:
# Function to load an extension
def load_extension(extension_name, extension_path):
    from pymrio.core.mriosystem import Extension
    
    ext = Extension(extension_name)
    
    # Load F matrix (industry accounts)
    F_path = os.path.join(extension_path, 'F.txt')
    if os.path.exists(F_path):
        ext.F = pd.read_csv(F_path, sep='\t', index_col=[0, 1], header=[0, 1])
        print(f"Loaded {extension_name} F matrix: {ext.F.shape}")
    
    # Load F_Y matrix (final demand accounts)
    F_Y_path = os.path.join(extension_path, 'F_Y.txt')
    if os.path.exists(F_Y_path):
        ext.F_Y = pd.read_csv(F_Y_path, sep='\t', index_col=[0, 1], header=[0, 1])
        print(f"Loaded {extension_name} F_Y matrix: {ext.F_Y.shape}")
    
    # Calculate extension accounts (multipliers and footprints)
    ext.calc_system(exio.x, exio.L, exio.Y)
    
    return ext

In [14]:
# Check the air emissions files
data_path = 'D:/Programming/databases/IOT_2022_pxp'
F_air = pd.read_csv(
    os.path.join(data_path, 'air_emissions', 'F.txt'),
    sep='\t',
    index_col=[0, 1],
    header=[0, 1]
)

In [15]:
print(f"\nAir emissions F shape: {F_air.shape}")
print(f"Number of sectors in F: {F_air.shape[1]}")
print(f"Number of sectors in x: {exio.x.shape[0]}")


Air emissions F shape: (418, 9799)
Number of sectors in F: 9799
Number of sectors in x: 9800


In [16]:
# Check which column is missing
print("\nColumn index comparison:")
print(f"F columns (first 5): {F_air.columns[:5].tolist()}")
print(f"x index (first 5): {exio.x.index[:5].tolist()}")


Column index comparison:
F columns (first 5): [('AT', 'Wheat'), ('AT', 'Cereal grains nec'), ('AT', 'Vegetables, fruit, nuts'), ('AT', 'Oil seeds'), ('AT', 'Sugar cane, sugar beet')]
x index (first 5): [('AT', 'Paddy rice'), ('AT', 'Wheat'), ('AT', 'Cereal grains nec'), ('AT', 'Vegetables, fruit, nuts'), ('AT', 'Oil seeds')]


In [17]:
# Find the missing sector
x_sectors = set(exio.x.index)
F_sectors = set(F_air.columns)

missing_in_F = x_sectors - F_sectors
missing_in_x = F_sectors - x_sectors

print(f"\nSectors in x but not in F: {missing_in_F}")
print(f"Sectors in F but not in x: {missing_in_x}")


Sectors in x but not in F: {('AT', 'Paddy rice')}
Sectors in F but not in x: set()


In [19]:
print(exio.material)

Extension material with parameters: name, F, F_Y, S, S_Y, M, D_cba, D_pba, D_imp, D_exp, unit, D_cba_reg, D_pba_reg, D_imp_reg, D_exp_reg


In [21]:
exio.material.get_rows()

Index(['Domestic Extraction Used - Primary Crops - Rice',
       'Domestic Extraction Used - Primary Crops - Wheat',
       'Domestic Extraction Used - Primary Crops - Cereals n.e.c.',
       'Domestic Extraction Used - Primary Crops - Roots and tubers',
       'Domestic Extraction Used - Primary Crops - Pulses',
       'Domestic Extraction Used - Primary Crops - Nuts',
       'Domestic Extraction Used - Primary Crops - Vegetables',
       'Domestic Extraction Used - Primary Crops - Fruits',
       'Domestic Extraction Used - Primary Crops - Oil bearing crops',
       'Domestic Extraction Used - Primary Crops - Sugar crops',
       'Domestic Extraction Used - Primary Crops - Fibres',
       'Domestic Extraction Used - Primary Crops - Spice - beverage - pharmaceutical crops',
       'Domestic Extraction Used - Primary Crops - Tobacco',
       'Domestic Extraction Used - Crop residues - Straw',
       'Domestic Extraction Used - Crop residues - Other crop residues (sugar and fodder beet 

In [23]:
print(exio)

IO System with parameters: Z, Y, x, A, L, unit, meta, air_emissions, employment, factor_inputs, material, nutrients, water


In [24]:
df1 = exio.F

AttributeError: 'IOSystem' object has no attribute 'F'